In [1]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re
import gc
import matplotlib.pyplot as plt
from tqdm import tqdm
api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

# 한글 폰트 설정
import matplotlib.font_manager as fm
import matplotlib as mpl

# 한글 폰트 경로 설정 (맥OS 기준)
font_path = '/System/Library/Fonts/AppleSDGothicNeo.ttc'  # 맥OS의 기본 한글 폰트
font_prop = fm.FontProperties(fname=font_path)

# matplotlib 기본 폰트 설정
plt.rc('font', family=font_prop.get_name())
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 폰트 확인
print(f"설정된 폰트: {font_prop.get_name()}")
print(f"사용 가능한 한글 폰트:")
for font in fm.findSystemFonts():
    if 'gothic' in font.lower() or 'gulim' in font.lower() or 'malgun' in font.lower() or 'batang' in font.lower():
        print(f" - {font}")

from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

df = pd.read_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', 
                 key='df')

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_61595/3294892761.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


설정된 폰트: Apple SD Gothic Neo
사용 가능한 한글 폰트:
 - /System/Library/Fonts/Supplemental/AppleGothic.ttf
 - /System/Library/Fonts/Supplemental/NotoSansGothic-Regular.ttf
 - /System/Library/Fonts/AppleSDGothicNeo.ttc


In [2]:
# pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', 100)
# pd.set_option('display.max_colwidth', None)

In [3]:
df[['환자번호',	
'날짜',
'End feel',
'CC_location',
'CC_pain_type',
'CC_painUncomp_desc_jaw',
'CC_disable_desc_jaw',
'CC_muscle_joint_desc_stress',
'CC_dentalHistory_desc',
'CC_clinic_history_desc',
'CC_factor_habbit',
'CC_treat_plan',
'CC_severity',
'CC_duration',
'약_medication_type',
'약_frequency',
'약_duration',
'약_compliance',
'장치_device_type',
'장치_usage_pattern',
'장치_duration',
'장치_compliance',
'습관_habit_type',
'습관_frequency',
'습관_awareness',
'습관_improvement',
'찜질_status',
'찜질_frequency',
'찜질_duration',
'찜질_method',
'마사지, 스트레칭_type',
'마사지, 스트레칭_frequency',
'마사지, 스트레칭_duration',
'마사지, 스트레칭_method',
'CMO_before',
'CMO_after',
'MMO_before',
'MMO_after',
'deviation_pattern_type',
'deviation_direction',
'deviation_intensity',
'Cap.pal_Pain_Intensity',
'Cap.pal_Pain_Direction',
'Cap.pal_Pain_Situation',
'M.pal_Pain_Intensity',
'M.pal_Pain_Direction',
'M.pal_Pain_Situation',
'Noise_Code',
'Noise_Direction',
'Noise_Intensity',
'Noise_Situation',
'Occlusion_lt_number',
'Occlusion_rt_number',
'Occlusion_lt_Intensity',
'Occlusion_rt_Intensity',
'oj',
'ob',
'Midline_Shift_Jaw',
'Midline_Shift_Direction_x',
'Midline_Shift_Direction_y',
'Midline_Shift_Amount',
'CRCO_Direction_x',
'CRCO_Direction_y',
'CRCO_Amount',
'Tongue_ridging_Intensity',
'Mucosal_ridging_Intensity',
'Rt_before',
'Rt_after',
'Lt_before',
'Lt_after',
'Next_Visit_Days',
'CC_vas',
'dif_MMO_CMO',
'첫방문_CMO_before',
'CMO_before_일일성장률',
'첫방문_CC_vas',
'CC_vas_일일성장률',
'첫방문_MMO_before',
'MMO_before_일일성장률',
'첫방문_dif_MMO_CMO',
'dif_MMO_CMO_일일성장률']].head()


,환자번호,날짜,End feel,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_duration,약_medication_type,약_frequency,약_duration,약_compliance,장치_device_type,장치_usage_pattern,장치_duration,장치_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,찜질_status,찜질_frequency,찜질_duration,찜질_method,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method",CMO_before,CMO_after,MMO_before,MMO_after,deviation_pattern_type,deviation_direction,deviation_intensity,Cap.pal_Pain_Intensity,Cap.pal_Pain_Direction,Cap.pal_Pain_Situation,M.pal_Pain_Intensity,M.pal_Pain_Direction,M.pal_Pain_Situation,Noise_Code,Noise_Direction,Noise_Intensity,Noise_Situation,Occlusion_lt_number,Occlusion_rt_number,Occlusion_lt_Intensity,Occlusion_rt_Intensity,oj,ob,Midline_Shift_Jaw,Midline_Shift_Direction_x,Midline_Shift_Direction_y,Midline_Shift_Amount,CRCO_Direction_x,CRCO_Direction_y,CRCO_Amount,Tongue_ridging_Intensity,Mucosal_ridging_Intensity,Rt_before,Rt_after,Lt_before,Lt_after,Next_Visit_Days,CC_vas,dif_MMO_CMO,첫방문_CMO_before,CMO_before_일일성장률,첫방문_CC_vas,CC_vas_일일성장률,첫방문_MMO_before,MMO_before_일일성장률,첫방문_dif_MMO_CMO,dif_MMO_CMO_일일성장률
0,2111-04,2021-11-11,soft,왼쪽 턱,통증,"턱 통증, 악관절에서 소리 남","제한된 개구, 턱 관절 움직임 제한","치아 악물기, 근육 긴장",Unknown,Unknown,Unknown,Unknown,3.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,10.0,Unknown,Unknown,Unknown,10.0,Unknown,27.0,Unknown,40.0,Unknown,other,unspecified,normal,0,unspecified,Unknown,0,both,Unknown,Click,left,0,Unknown,0,0,0,0,2.0,2.0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,0,Unknown,0.73,Unknown,0.73,Unknown,3.0,13.0,27.0,0.000000,3.0,0.0,40.0,0.0,13.0,0.000000
1,2111-04,2021-11-25,soft,왼쪽 턱,뚜둑뚜둑 걸리는 느낌,턱에서 소리가 났어요,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,3.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,10.0,Unknown,Unknown,Unknown,10.0,Unknown,27.0,Unknown,44.0,Unknown,other,unspecified,normal,0,unspecified,Unknown,0,both,Unknown,Click,left,0,벌릴 때,0,0,0,0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,0,Unknown,Unknown,Unknown,Unknown,Unknown,3.0,17.0,27.0,0.000000,3.0,0.0,40.0,10.0,13.0,30.769231
2,2111-04,2021-12-09,soft,왼쪽 턱,통증,턱이 빠지는 느낌,Unknown,Unknown,Unknown,Unknown,찜질 1주일에 3번 정도는 했어요. 일부러 입 크게 안 벌리는 등 습관조절 노력했어요.,Unknown,3.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,10.0,Unknown,Unknown,Unknown,10.0,Unknown,30.0,Unknown,45.0,Unknown,other,unspecified,normal,0,unspecified,Unknown,0,unspecified,Unknown,Click,left,0,벌릴 때,0,0,0,0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,0,Unknown,Unknown,Unknown,Unknown,Unknown,3.0,15.0,27.0,11.111111,3.0,0.0,40.0,12.5,13.0,15.384615
3,2111-04,2022-08-30,soft,왼쪽 턱,통증,삐그덕거리고,입 크게 안 벌리는 습관조절,Unknown,Unknown,Unknown,Unknown,Unknown,3.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,10.0,Unknown,Unknown,Unknown,10.0,Unknown,38.0,Unknown,48.0,Unknown,other,unspecified,normal,0,unspecified,Unknown,0,unspecified,Unknown,No-Noise,none,0,Unknown,0,0,0,0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,0,Unknown,Unknown,Unknown,Unknown,14.0,3.0,10.0,27.0,40.740741,3.0,0.0,40.0,20.0,13.0,-23.076923
4,2111-07,2021-11-11,soft,턱관절 쪽,통증,통증,제한된 개구,Unknown,Unknown,Unknown,Unknown,Unknown,2.0,10.0,Unknown,Unknown,NaN,Unknown,Unknown,Unknown,2.0,Unknown,치아 접촉 줄이기,Unknown,Unknown,Unknown,Unknown,Unknown,10.0,Unknown,Unknown,Unknown,10.0,Unknown,35.0,Unknown,43.0,Unknown,other,unspecified,normal,0,unspecified,Unknown,0,left,Unknown,No-Noise,none,0,Unknown,0,0,0,0,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,0,Unknown,Unknown,Unknown,Unknown,7.0,2.0,8.0,35.0,0.